<a href="https://colab.research.google.com/github/zhilyaevaviktorija/machine_learning/blob/main/homeworks/Homework_8_CNN/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Обучение CNN на собранном корпусе данных**

#### **Задача 1.** Собрать свой текстовый корпус (не менее 500 примеров) с помощью библиотеки requests или другого инструмента парсинга

Код брала с сайта https://sky.pro/wiki/python/kak-parsit-dannye-s-sajta-s-pomoshyu-python/ и потом модифицировала его, чтобы собрать минимум 500 примеров. Сайт для парсинга был взят из кода.



In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

def parse_hacker_news_page(page_num):
    """Парсинг одной страницы Hacker News"""
    if page_num == 1:
        url = 'https://news.ycombinator.com/'
    else:
        url = f'https://news.ycombinator.com/news?p={page_num}'

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    news_items = soup.select('.athing')
    page_data = []

    for item in news_items:
        title_elem = item.select_one('.titleline > a')
        if title_elem:
            title = title_elem.text
            link = title_elem.get('href')

            # Получаем ID для поиска скора
            item_id = item.get('id')
            score_elem = soup.select_one(f'#score_{item_id}')
            score = score_elem.text if score_elem else "0 points"

            page_data.append({
                'title': title,
                'link': link,
                'score': score
            })

    return page_data

# Собираем данные со всех доступных страниц
all_news = []
max_pages = 28  # Максимальное количество страниц

for page in range(1, max_pages + 1):
    print(f"Парсинг страницы {page}...")
    news = parse_hacker_news_page(page)
    all_news.extend(news)
    print(f"Добавлено {len(news)} новостей. Всего: {len(all_news)}")
    time.sleep(random.uniform(1, 2))

df = pd.DataFrame(all_news)
df.to_csv('hacker_news_data.csv', index=False)
print(f"Итого собрано: {len(df)} новостей")
print("Данные сохранены в 'hacker_news_data.csv'")

# Максимально удалось спарсить 840 новостей с 28 страниц

Парсинг страницы 1...
Добавлено 30 новостей. Всего: 30
Парсинг страницы 2...
Добавлено 30 новостей. Всего: 60
Парсинг страницы 3...
Добавлено 30 новостей. Всего: 90
Парсинг страницы 4...
Добавлено 30 новостей. Всего: 120
Парсинг страницы 5...
Добавлено 30 новостей. Всего: 150
Парсинг страницы 6...
Добавлено 30 новостей. Всего: 180
Парсинг страницы 7...
Добавлено 30 новостей. Всего: 210
Парсинг страницы 8...
Добавлено 30 новостей. Всего: 240
Парсинг страницы 9...
Добавлено 30 новостей. Всего: 270
Парсинг страницы 10...
Добавлено 30 новостей. Всего: 300
Парсинг страницы 11...
Добавлено 30 новостей. Всего: 330
Парсинг страницы 12...
Добавлено 30 новостей. Всего: 360
Парсинг страницы 13...
Добавлено 30 новостей. Всего: 390
Парсинг страницы 14...
Добавлено 30 новостей. Всего: 420
Парсинг страницы 15...
Добавлено 30 новостей. Всего: 450
Парсинг страницы 16...
Добавлено 30 новостей. Всего: 480
Парсинг страницы 17...
Добавлено 30 новостей. Всего: 510
Парсинг страницы 18...
Добавлено 30 новосте

In [ ]:
df = pd.read_csv('hacker_news_data.csv')
df

,title,link,score
0,US bans differential privacy in Census data,https://desfontain.es/blog/banning-noise.html,405 points
1,Treating pancreatic tumours may have revealed ...,https://economist.com/science-and-technology/2...,146 points
2,Show HN: Verso – A $14.99 Mac word processor w...,https://www.versowriter.app,25 points
3,Every Frame Perfect,https://tonsky.me/blog/every-frame-perfect/,265 points
4,Appreciating Exif,https://brentfitzgerald.com/posts/appreciating...,66 points
...,...,...,...
835,All the Ways Europe Is Ditching American Techn...,https://www.wired.com/story/all-the-ways-europ...,32 points
836,Donut Lab's solid-state battery claim debunked...,https://www.theverge.com/science/946608/donut-...,16 points
837,"Reeed – a read-it-later app for iOS, built aft...",https://www.reeed.io/,5 points
838,SAT-Physical Thermodynamic Framework: treating...,https://github.com/alikamp/SAT_HARDNESS_P-NP,12 points


#### **Задача 2.** Выполнить разметку собранных данных для задачи классификации (бинарной или многоклассовой)

Для разметки данных использую бинарную классификацию по столбцу score (рейтинг новости):
-   Класс 0 (low_score) — новости с рейтингом ≤ 100 очков
-   Класс 1 (high_score) — новости с рейтингом > 100 очков

In [ ]:
import pandas as pd

# Функция для извлечения числового значения из строки типа "182 points"
def extract_score(score_str):
    """Преобразует '182 points' → 182"""
    try:
        return int(score_str.split()[0])
    except:
        return 0

# Применяем функцию к столбцу score
df['score_numeric'] = df['score'].apply(extract_score)

# Создаем метку класса: 1 если рейтинг > 100, иначе 0
df['label'] = (df['score_numeric'] > 100).astype(int)

# Проверяем баланс классов
print("Распределение классов:")
print(df['label'].value_counts())
print(f"\nКласс 1 (популярные, >100 очков): {df['label'].sum()}")
print(f"Класс 0 (обычные, ≤100 очков): {len(df) - df['label'].sum()}")

# Сохраняем размеченные данные
df.to_csv('hacker_news_labeled.csv', index=False)
print("\nРазмеченные данные сохранены в 'hacker_news_labeled.csv'")

# Показываем примеры
print("\nПримеры размеченных новостей:")
print(df[['title', 'score', 'score_numeric', 'label']].head(10))

Распределение классов:
label
0    548
1    292
Name: count, dtype: int64

Класс 1 (популярные, >100 очков): 292
Класс 0 (обычные, ≤100 очков): 548

Размеченные данные сохранены в 'hacker_news_labeled.csv'

Примеры размеченных новостей:
                                               title       score  \
0        US bans differential privacy in Census data  405 points   
1  Treating pancreatic tumours may have revealed ...  146 points   
2  Show HN: Verso – A $14.99 Mac word processor w...   25 points   
3                                Every Frame Perfect  265 points   
4                                  Appreciating Exif   66 points   
5  Introduction to the experience of rendering Ar...   96 points   
6  The adder at the heart of Intel's 8087 floatin...   27 points   
7  A low-carbon computing platform from your reti...  184 points   
8                                     GLM 5.2 Is Out   84 points   
9  AI OSS tool repo goes archived over night afte...  192 points   

   score_numeri

Краткие выводы по задаче 2:
-   собранный датасет получился несложным без нескольких классов, поэтому бинарная классификация была выбрана для простоты;
-   порог 100 взят как медианное значение score;
-   данная автоматическая разметка не требует ручной проверки.

#### **Задача 3.** Обучить свёрточную нейросеть (CNN) на собранных и размеченных данных с использованием PyTorch или TensorFlow на выбор

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense

# 1. Загрузка размеченных данных
df = pd.read_csv('hacker_news_labeled.csv')
df

,title,link,score,score_numeric,label
0,US bans differential privacy in Census data,https://desfontain.es/blog/banning-noise.html,405 points,405,1
1,Treating pancreatic tumours may have revealed ...,https://economist.com/science-and-technology/2...,146 points,146,1
2,Show HN: Verso – A $14.99 Mac word processor w...,https://www.versowriter.app,25 points,25,0
3,Every Frame Perfect,https://tonsky.me/blog/every-frame-perfect/,265 points,265,1
4,Appreciating Exif,https://brentfitzgerald.com/posts/appreciating...,66 points,66,0
...,...,...,...,...,...
835,All the Ways Europe Is Ditching American Techn...,https://www.wired.com/story/all-the-ways-europ...,32 points,32,0
836,Donut Lab's solid-state battery claim debunked...,https://www.theverge.com/science/946608/donut-...,16 points,16,0
837,"Reeed – a read-it-later app for iOS, built aft...",https://www.reeed.io/,5 points,5,0
838,SAT-Physical Thermodynamic Framework: treating...,https://github.com/alikamp/SAT_HARDNESS_P-NP,12 points,12,0


In [ ]:
# 2. Предобработка

def clean_text(text):
    text = text.lower()                  # приводим к нижнему регистру
    text = re.sub(r'[^\w\s]', '', text)  # убираем пунктуацию
    text = re.sub(r'\d+', '', text)      # убираем цифры
    return text.strip()

df['clean_title'] = df['title'].apply(clean_text)
df

,title,link,score,score_numeric,label,clean_title
0,US bans differential privacy in Census data,https://desfontain.es/blog/banning-noise.html,405 points,405,1,us bans differential privacy in census data
1,Treating pancreatic tumours may have revealed ...,https://economist.com/science-and-technology/2...,146 points,146,1,treating pancreatic tumours may have revealed ...
2,Show HN: Verso – A $14.99 Mac word processor w...,https://www.versowriter.app,25 points,25,0,show hn verso a mac word processor with no s...
3,Every Frame Perfect,https://tonsky.me/blog/every-frame-perfect/,265 points,265,1,every frame perfect
4,Appreciating Exif,https://brentfitzgerald.com/posts/appreciating...,66 points,66,0,appreciating exif
...,...,...,...,...,...,...
835,All the Ways Europe Is Ditching American Techn...,https://www.wired.com/story/all-the-ways-europ...,32 points,32,0,all the ways europe is ditching american techn...
836,Donut Lab's solid-state battery claim debunked...,https://www.theverge.com/science/946608/donut-...,16 points,16,0,donut labs solidstate battery claim debunked b...
837,"Reeed – a read-it-later app for iOS, built aft...",https://www.reeed.io/,5 points,5,0,reeed a readitlater app for ios built after p...
838,SAT-Physical Thermodynamic Framework: treating...,https://github.com/alikamp/SAT_HARDNESS_P-NP,12 points,12,0,satphysical thermodynamic framework treating c...


In [ ]:
# 3. Токенизация
tokenizer = Tokenizer(num_words=3000)
tokenizer.fit_on_texts(df['clean_title'])
X = pad_sequences(tokenizer.texts_to_sequences(df['clean_title']), maxlen=50)
y = df['label'].values
X.shape

(840, 50)

In [ ]:
# 4. Делим данные на train/val/test выборки
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Train: 588, Val: 126, Test: 126


#### **Задача 4.** Самостоятельно подобрать архитектуру модели и гиперпараметры (количество слоёв, размер ядра, функцию активации, оптимизатор и т.д.)

In [ ]:
# 5. Создаем архитектуру модели CNN
from tensorflow.keras.layers import Dropout

model = Sequential([
    Embedding(3000, 32),              # эмбеддинги слов
    Conv1D(32, 3, activation='relu'), # извлекаем n-граммы
    GlobalMaxPooling1D(),             # свёртываем в векторы
    Dense(16, activation='relu'),     # полносвязный слой
    Dropout(0.5),                     # для регуляции
    Dense(1, activation='sigmoid')    # выходной слой (бинарная классификация)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Модель создана")

Модель создана


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,                         # больше эпох
    batch_size=32,                     # увеличила батч
    class_weight=class_weight_dict,    # ручные веса
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6071 - loss: 0.6944 - val_accuracy: 0.6270 - val_loss: 0.6892
Epoch 2/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6020 - loss: 0.6828 - val_accuracy: 0.4444 - val_loss: 0.6957
Epoch 3/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6582 - loss: 0.6684 - val_accuracy: 0.3968 - val_loss: 0.6988
Epoch 4/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7857 - loss: 0.6401 - val_accuracy: 0.4921 - val_loss: 0.6944
Epoch 5/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8486 - loss: 0.6062 - val_accuracy: 0.5000 - val_loss: 0.6956
Epoch 6/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9337 - loss: 0.5328 - val_accuracy: 0.5079 - val_loss: 0.6946
Epoch 7/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9405 - loss: 0.4534 - val_accuracy: 0.4841 - val_loss: 0.6957
Epoch 8/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9677 - loss: 0.3546 - val_accuracy: 0.4921 - val_loss

In [ ]:
# 7. Сохраняем модель
model.save('hacker_news_cnn_model.keras')
print("Модель сохранена")

Модель сохранена


#### **Задача 5.** После обучения вывести на экран метрики качества (accuracy, precision, recall, F1-score) на тестовой выборке

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Предсказания на тестовой выборке
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Метрики
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nAccuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

# после добавления Dropout(0.5) в архитектуру модели метрики показывают более низкие результаты

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

Accuracy:  0.5873
Precision: 0.2778
Recall:    0.1136
F1-score:  0.1613


In [ ]:
# Выводим метрики в виде таблицы

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low score (≤100)', 'High score (>100)']))

Classification Report:
                   precision    recall  f1-score   support

 Low score (≤100)       0.64      0.84      0.73        82
High score (>100)       0.28      0.11      0.16        44

         accuracy                           0.59       126
        macro avg       0.46      0.48      0.44       126
     weighted avg       0.51      0.59      0.53       126



In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Модель плохо предсказывает классы

Confusion Matrix:
[[69 13]
 [39  5]]


#### **Выбор гиперпараметров**

Я выбрала:
1. размер эмбеддингов = 32, потому что при небольшом датасете происходит переобучение при увеличении размера эмбеддингов;

2. Conv1D = 32, потому что этого достаточно для выделения базовых n-грамм, иначе будет перебучение;

3. kernel_size=3, потому что этого достаточно для выделения биграмм и триграмм;

4. Dropout = 0.5, потому что этот параметр помогает снижать переобучение;

5. Batch size = 32, потому что при 16 была нестабильная сходимость, а при 64 - медленная сходимость.

#### **Анализ результатов**

Модель хорошо работает, обучение завершено без ошибок. В коде указаны все этапы из задания. Удалось собрать 840 размеченных примеров с Hacker News.

Модель недостаточно хорошо классифицирует примеры, т.е. она часто ошибается при предсказывании класса 1. Скорее всего это связано с небольшим датасетом. Возможно стоило выбрать многоклассовую разметку текста или просто увеличить датасет.